---

# Mapa SPI por Mês para 2024

---

- `OBJETIVO`:
> Este código calcula e plota o mapa de SPI para cada mês de 2024 com dados do CPC para o Estado de SP.



- `DADOS DE ENTRADA`:
> Dados globais diários de precipitação do Climate Prediction Center (CPC) com 50km de resolução espacial. Os dados são disponíveis desde 1979 e são carregados diretamente através do Xarray via Protocolo OPeNDAP através do seguinte [link](http://apdrc.soest.hawaii.edu:80/dods/public_data/Interpolated_precipitation/cpc_rainfall/Daily). Mais informações dos dados acesse [aqui](http://apdrc.soest.hawaii.edu/dods/public_data/Interpolated_precipitation/cpc_rainfall/Daily.info) e diversos outros dados disponibilizados pelo Asia-Pacific Data Research Center, acesse [aqui](https://apdrc.soest.hawaii.edu/data/data.php?discipline_index=3).


- `DADOS DE SAÍDA`:
> 1. Mapa em formato JPG do SPI mensal para 2024. Exemplo: `01_SPI_12_mensal_2024.jpg`


- `OBSERVAÇÕES`:
   > Nenhuma.

- `REALIZADO POR`:
> Prof. Enrique V. Mattos / UNIFEI - 05/06/2026

- `ATUALIZADO POR`:
> Prof. Enrique V. Mattos / UNIFEI - 05/06/2026
---


# Prepando ambiente


In [1]:
#=========================================================================================================================#
#                                          INSTALAÇÃO E IMPORTAÇÃO DAS BIBLIOTECAS
#=========================================================================================================================#
# instala bibliotecas
!pip install -q xarray dask netCDF4 bottleneck xclim ultraplot cartopy salem rasterio pyproj geopandas geobr

# importa bibliotecas
import ultraplot as uplt
import geobr
import numpy as np
import time
import xarray as xr
import xclim as xc
from xclim.indices import standardized_precipitation_index
from datetime import datetime
import ultraplot as uplt
import salem
import os
import cartopy.io.shapereader as shpreader
import cartopy.crs as ccrs
from cartopy.mpl.patch import geos_to_path
from matplotlib.path import Path
from matplotlib.patches import PathPatch
import matplotlib.pyplot as plt
from matplotlib.colors import BoundaryNorm, ListedColormap
from matplotlib.cm import ScalarMappable
from cartopy import crs as ccrs
import warnings
warnings.filterwarnings("ignore")

#=========================================================================================================================#
#                                  MONTA O GOOGLE DRIVE E CRIA O DIRETÓRIO DE SAÍDA
#=========================================================================================================================#
# monta o drive
from google.colab import drive
drive.mount('/content/drive')

# diretório raiz
dir = '/content/drive/MyDrive/PYHTON/00_GITHUB/000_CODIGOS_REFERENCIA/08_SPI'

# diretório de saída
dir_output = f'{dir}/output'

# cria pasta de saída
os.makedirs(dir_output, exist_ok=True)

#=========================================================================================================================#
#                                  DEFINE OS LIMITES DO BRASIL OU ESTADO DE SÃO PAULO
#=========================================================================================================================#
# limites
lonmin, lonmax, latmin, latmax = -53.3, -43.9, -25.4, -19.7 # estado de SP
#lonmin, lonmax, latmin, latmax = -75.0, -33.0, -35.0, 7.0 # Brasil

# leitura shapefiles
shapefile = salem.read_shapefile('https://github.com/evmpython/shapefile/raw/main/UFs/SP/SP_UF_2019.shp') # estado de SP
#shapefile = salem.read_shapefile('https://github.com/evmpython/shapefile/raw/main/brasil/BRAZIL.shp') # Brasil

# é importante porque o NetCDF usa lon/lat e o Cartopy também vai plotar em PlateCarree
shapefile = shapefile.to_crs("EPSG:4326") # Pois os dois tem que se conversar

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 51.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 384.7/384.7 kB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 54.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.8/11.8 MB 35.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.1/86.1 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 338.0/338.0 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.2/194.2 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.4/79.4 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 41.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 43.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 307.5/307.5 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 83.5/83.5 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 

Mounted at /content/drive


# Plota Mapa

In [ ]:
%%time
#=========================================================================================================================#
#                                            CARREGA OS DADOS DO CPC
#=========================================================================================================================#
# carrega os dados
cpc_precip_mensal = xr.open_dataset('http://apdrc.soest.hawaii.edu:80/dods/public_data/Interpolated_precipitation/cpc_rainfall/0.5deg').squeeze()

# transforma as longitudes de 0/360 para -180/+180
cpc_precip_mensal.coords['lon'] = ((cpc_precip_mensal.coords['lon'] + 180) % 360) - 180 ; cpc_precip_mensal = cpc_precip_mensal.sortby(cpc_precip_mensal.lon)

# datas
data_inicial, data_final = '1980-01-01', '2024-12-31'

# seleciona os dados
cpc_precip_mensal = cpc_precip_mensal.sel(lon=slice(lonmin,lonmax), lat=slice(latmin,latmax)).sel(time=slice(data_inicial, data_final))

#=========================================================================================================================#
#                                                 CALCULA O SPI
#=========================================================================================================================#
# define qual SPI [1, 3, 6, 12]
spi_window = 3

# define o ano de interesse
ano_escolhido = '2024'

# extrai a precipitação
pr = cpc_precip_mensal.rain

# define a unidade
pr.attrs['units'] = 'mm/month'

# define o nome da variável
pr.attrs['standard_name'] = 'precipitation_amount'

# define o período de calibração adequado (mínimo 30 anos)
cal_start_date = datetime(1980, 1, 1)  # início dos dados
cal_end_date = datetime(2020, 12, 31)  # 40 anos de dados

# calcula o SPI
spi = standardized_precipitation_index(pr,
                                       freq="MS",
                                       window=spi_window,
                                       dist="gamma",
                                       method="ML",
                                       cal_start=cal_start_date,
                                       cal_end=cal_end_date)

# extrai o SPI do ano escolhido
spi_ano_escolhido = spi.sel(time=f'{ano_escolhido}')

# recorta para região
nome = 'São Paulo'
shapefiles = geobr.read_state(year=2020)
shapefile = shapefiles[shapefiles['name_state'] == nome]
spi_ano_escolhido = spi_ano_escolhido#.salem.roi(shape=shapefile)

# verifica se o SPI foi calculado corretamente
print(f"SPI {spi_window} - Valores NaN: {np.isnan(spi).sum().values}")
print(f"SPI {spi_window} - Valores válidos: {spi.notnull().sum().values}")

#=========================================================================================================================#
#                                                 PLOTA MAPA DE SPI
#=========================================================================================================================#
#-----------------------------------------------------------#
#               DEFINIÇÕES DA COLORMAP
#-----------------------------------------------------------#
# define os níveis (limites dos intervalos)
levels = [-3.0, -2.0, -1.5, -1.0, 0, 1.0, 1.5, 2.0, 3.0]

# define os rótulos para cada intervalo
labels = ['Extremamente seco',    # < -2.0
          'Severamente seco',     # -2.0 a -1.5
          'Moderamente Seco',     # -1.5 a -1.0
          'Normal',               # -1.0 a 0
          'Normal',               # 0 a 1.0
          'Moderamente úmido',    # 1.0 a 1.5
          'Severamente úmido',    # 1.5 a 2.0
          'Extremamente úmido'    # >= 2.0
          ]

# cria a colormap
n_colors = len(levels) - 1
base_cmap = plt.cm.get_cmap('coolwarm', n_colors)
new_colors = base_cmap(np.arange(n_colors))

# reverte a colormap para obter coolwarm_r (vermelho para o azul)
new_colors = new_colors[::-1]

# define azul e vermelho leve
light_red_color = (1.0, 0.7, 0.7, 1.0)  # vermelho leve
light_blue_color = (0.7, 0.7, 1.0, 1.0) # azul leve

# aloca as cores da classe 'Normal'
# For [-1.0, 0) (index 3) use light red
new_colors[3] = light_red_color
# For [0, 1.0) (index 4) use light blue
new_colors[4] = light_blue_color

# cria a colormap customizada
custom_cmap = ListedColormap(new_colors)

# cria a normalização para os níveis discretos
norm = BoundaryNorm(levels, ncolors=n_colors)

# usa o custom_cmap para o mapa
cmap = custom_cmap

#-----------------------------------------------------------#
#               PARÂMETROS DA FIGURA
#-----------------------------------------------------------#
# cria a moldura da figura
fig, ax = uplt.subplots(axheight=3,
                        nrows=3, ncols=4,
                        tight=True,
                        proj='pcarree',
                        sharex=True, sharey=True)

# formatação dos eixos
ax.format(coast=False, borders=False, innerborders=False,
          labels=False, latlines=5, lonlines=10,
          latlim=(latmin, latmax), lonlim=(lonmin, lonmax),
          suptitle=f'SPI-{spi_window}: 2024',
          small='20px', large='25px',
          linewidth=0, grid=False, abc=False)

# meses
meses = ['Janeiro', 'Fevereiro', 'Março', 'Abril', 'Maio', 'Junho',
         'Julho', 'Agosto', 'Setembro', 'Outubro', 'Novembro', 'Dezembro']

#-----------------------------------------------------------#
#                    PLOTA FIGURA
#-----------------------------------------------------------#
# loop dos meses
for i, mes in enumerate(meses):

    # exibe na tela o mes que esta sendo processado
    print(f'Processando o mês: .... {i} {mes}')

    # plota figura com os níveis personalizados
    map1 = ax[i].contourf(spi_ano_escolhido['lon'],
                          spi_ano_escolhido['lat'],
                          spi_ano_escolhido[i,:,:],
                          cmap=cmap, # Usando o colormap customizado
                          norm=norm,
                          levels=levels,
                          extend='both')  # estende para valores abaixo/acima dos níveis

    # suaviza as bordas
    geom = shapefile.geometry.unary_union
    minx, miny, maxx, maxy = shapefile.total_bounds
    paths = geos_to_path([geom])
    clip_path = Path.make_compound_path(*paths)
    clip_patch = PathPatch(clip_path, transform=ax[i].transData, facecolor="none")
    map1.set_clip_path(clip_patch)

    # plota titulo de cada figura
    ax[i].format(title=str(mes), labels=False, titleloc='c',
                 titlecolor='grey', titleweight='bold', titlesize=20)

    # shapefile da região
    shapefile_plot = list(shpreader.Reader('https://github.com/evmpython/shapefile/raw/main/UFs/SP/SP_UF_2019.shp').geometries())
    ax[i].add_geometries(shapefile_plot, ccrs.PlateCarree(), edgecolor='black', facecolor='none', linewidth=1.0)

#-----------------------------------------------------------#
#          BARRA DE CORES PERSONALIZADA COM NOMES
#-----------------------------------------------------------#
# cria um ScalarMappable com a normalização e o mapa de cores
sm = ScalarMappable(norm=norm, cmap=cmap)
sm.set_array(np.array(levels))

# calcula a localização dos labels (pontos médios dos intervalos)
tick_locations = [(levels[i] + levels[i+1]) / 2 for i in range(len(levels) - 1)]

# combina os labels numéricos e descritivos
combined_labels = []
combined_labels.append(f'< {levels[1]:.1f}: {labels[0]}') # Extremamente seco: < -2.0
combined_labels.append(f'[{levels[1]:.1f} à {levels[2]:.1f}]: {labels[1]}') # Severamente seco: [-2.0, -1.5)
combined_labels.append(f'[{levels[2]:.1f} à {levels[3]:.1f}]: {labels[2]}') # Moderamente Seco: [-1.5, -1.0)
combined_labels.append(f'[{levels[3]:.1f} à {levels[4]:.1f}]: {labels[3]}') # Normal: [-1.0, 0.0) (Light Red)
combined_labels.append(f'[{levels[4]:.1f} à {levels[5]:.1f}]: {labels[4]}') # Normal: [0.0, 1.0) (Light Blue)
combined_labels.append(f'[{levels[5]:.1f} à {levels[6]:.1f}]: {labels[5]}') # Moderamente úmido: [1.0, 1.5)
combined_labels.append(f'[{levels[6]:.1f} à {levels[7]:.1f}]: {labels[6]}') # Severamente úmido: [1.5, 2.0)
combined_labels.append(f'>= {levels[7]:.1f}: {labels[7]}') # Extremamente úmido: >= 2.0

# adiciona a barra de cores com os rótulos personalizados
cbar = fig.colorbar(sm,
                    loc='r',
                    ticks=tick_locations,
                    ticklabelsize=20,
                    labelsize=20,
                    length=0.605,
                    width=0.40,
                    space=0.3)

# define os rótulos da barra de cores
cbar.ax.set_yticklabels(combined_labels)

# ajusta o espaçamento dos rótulos para evitar sobreposição
cbar.ax.tick_params(axis='y', which='major', pad=10)

#-----------------------------------------------------------#
#                  SALVA FIGURA
#-----------------------------------------------------------#
fig.save(f'{dir_output}/01_SPI_{spi_window}_mensal_2024.jpg', bbox_inches='tight', dpi=300)

SPI 1 - Valores NaN: 0
SPI 1 - Valores válidos: 123120
Processando o mês: .... 0 Janeiro
Processando o mês: .... 1 Fevereiro
Processando o mês: .... 2 Março
Processando o mês: .... 3 Abril


In [3]:
# mostra os dados de SPI
spi

<xarray.DataArray 'rain' (time: 540, lat: 12, lon: 19)> Size: 985kB
array([[[        nan,         nan,         nan, ...,         nan,
                 nan,         nan],
        [        nan,         nan,         nan, ...,         nan,
                 nan,         nan],
        [        nan,         nan,         nan, ...,         nan,
                 nan,         nan],
        ...,
        [        nan,         nan,         nan, ...,         nan,
                 nan,         nan],
        [        nan,         nan,         nan, ...,         nan,
                 nan,         nan],
        [        nan,         nan,         nan, ...,         nan,
                 nan,         nan]],

       [[        nan,         nan,         nan, ...,         nan,
                 nan,         nan],
        [        nan,         nan,         nan, ...,         nan,
                 nan,         nan],
        [        nan,         nan,         nan, ...,         nan,
                 nan,         nan],
...
        [-2.07477322, -1.37059916, -1.12453812, ..., -1.02156968,
         -1.16475676, -0.60843195],
        [-1.73539248, -1.10607573, -0.95244253, ..., -1.18292298,
         -1.19855942, -1.20827051],
        [-1.21485705, -0.8824456 , -0.82866379, ..., -0.96669169,
         -1.02063485, -0.94873072]],

       [[-0.47394757, -0.45189697, -0.50224342, ..., -0.32867963,
         -0.35052996, -0.30421491],
        [-0.60783698, -0.57321782, -0.65921545, ..., -0.36710379,
         -0.35361838, -0.28857197],
        [-0.70808814, -0.72309794, -0.8978407 , ..., -0.37158671,
         -0.32383963, -0.25973604],
        ...,
        [-1.37558563, -1.06012469, -0.88164009, ..., -0.28050078,
         -0.26895367, -0.2781243 ],
        [-1.2821474 , -0.87603985, -0.73913949, ..., -0.24635617,
         -0.23941502, -0.26210087],
        [-0.93781363, -0.70116734, -0.67025869, ..., -0.15731112,
         -0.15082636, -0.18679679]]])
Coordinates:
  * time               (time) datetime64[ns] 4kB 1980-01-01 ... 2024-12-01
  * lat                (lat) float64 96B -25.25 -24.75 -24.25 ... -20.25 -19.75
  * lon                (lon) float64 152B -53.25 -52.75 -52.25 ... -44.75 -44.25
    lev                float64 8B 1.0
    number_of_zeros    (time, lat, lon) int64 985kB 0 0 0 0 0 0 ... 0 0 0 0 0 0
    number_of_notnull  (time, lat, lon) int64 985kB 40 40 40 40 ... 41 41 41 41
    prob_of_zero       (time, lat, lon) float64 985kB 0.0 0.0 0.0 ... 0.0 0.0
Attributes:
    calibration_period:  ('1980-01-01', '2020-12-01')
    freq:                MS
    window:              12
    scipy_dist:          gamma
    method:              ML
    group:               time.month
    units:               1
    time_indexer:        {}